# 00 Pull Raw Statcast Data

Pull the 2025 top-100 pitcher panel from the Statcast arsenal leaderboard and cache raw pitch-level data.
This notebook is intentionally about inputs and coverage, not modeling.

In [1]:
# Environment setup and helper utility definitions
import os
import sys
import subprocess
from pathlib import Path

# Detect if running in Google Colab for specific dependency and path handling
IN_COLAB = "google.colab" in sys.modules
REPO_URL = ""  # Target repository for cloning source code if needed
TOP_N = 100

if IN_COLAB:
    # Install core baseball analytics libraries
    %pip -q install -r requirements.txt || %pip -q install pybaseball pandas numpy scikit-learn matplotlib seaborn pyarrow shap statsmodels xgboost tqdm
    # Clone the project repository if a URL is provided and it doesn't exist locally
    if REPO_URL and not Path("/content/pitch-sequencing").exists():
        !git clone {REPO_URL} /content/pitch-sequencing
    BASE_DIR = Path("/content/pitch-sequencing") if Path("/content/pitch-sequencing").exists() else Path.cwd()
else:
    BASE_DIR = Path.cwd()

os.chdir(BASE_DIR)
sys.path.insert(0, str(BASE_DIR / "code"))

# defining data and output pipelines
DATA_DIR = BASE_DIR / "data"
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"
OUTPUT_DIR = BASE_DIR / "output"

# folder structure/making directory
for path in [RAW_DIR, PROCESSED_DIR, OUTPUT_DIR, OUTPUT_DIR / "figures"]:
    path.mkdir(parents=True, exist_ok=True)

def run_step(args):
    """Executes shell commands/scripts within the pipeline context and captures logs."""
    print("Running:", " ".join(map(str, args)))
    result = subprocess.run(args, cwd=BASE_DIR, text=True, capture_output=True)
    print(result.stdout)
    if result.stderr:
        print(result.stderr)
    result.check_returncode()

def frame_diag(df, label):
    """Prints summary statistics and distributions for a DataFrame to verify data integrity."""
    print(f"{label}: rows={len(df):,}, cols={df.shape[1]:,}")
    if "pitcher_name" in df:
        print(f"{label}: pitchers={df['pitcher_name'].nunique():,}")
    if "pitch_type" in df:
        print(f"{label}: pitch_types={df['pitch_type'].nunique():,}")
        print(df["pitch_type"].value_counts().head(12))
    return df.head()

def merge_diag(left, right, keys, label):
    """Performs a validated left merge and reports on the success rate and potential duplicates."""
    print(f"[merge:{label}] before left_rows={len(left):,}, right_rows={len(right):,}, keys={keys}")
    print(f"[merge:{label}] right_duplicate_keys={right.duplicated(keys).sum():,}")
    merged = left.merge(right, on=keys, how="left", validate="many_to_one", indicator=True)
    print(f"[merge:{label}] after rows={len(merged):,}, unmatched={merged['_merge'].eq('left_only').sum():,}")
    return merged.drop(columns=["_merge"])

def show_csv(path, rows=8):
    """Utility to quickly load and display a CSV file from the project directory."""
    import pandas as pd
    path = BASE_DIR / path
    print(path)
    df = pd.read_csv(path)
    display(df.head(rows))
    return df

ERROR: Could not open requirements file: [Errno 2] No such file or directory: 'requirements.txt'
/bin/bash: line 1: fg: no job control


In [ ]:
from pitch_sequence_pipeline import (
    cache,
    ensure_dirs,
    pull_pitcher_data,
    select_pitchers,
)

ensure_dirs()
cache.enable()

pitchers = select_pitchers(2025, TOP_N)
frame_diag(pitchers, "selected_pitchers")
display(pitchers.head(15))

raw_2025 = pull_pitcher_data(
    season=2025,
    start_date="2025-03-18",
    end_date="2025-09-28",
    pitchers=pitchers,
    raw_label="statcast",
)
frame_diag(raw_2025, "raw_2025")

Outputs: `output/top_100_pitchers_2025.csv` and cached pitcher parquet files in `data/raw/`.